# 06a · Phase 3: YOLOv26s Shard

Trains 6 loss configs × 3 splits = **18 runs** with YOLOv26s.  
Est. runtime: **~1.5 h** on Kaggle T4.

Part of the Phase 3 cross-architecture study. Run in parallel with 06b and 06c.

## Section 1 · Environment

In [ ]:
# Environment detection
import os, sys
from pathlib import Path

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    ROOT = Path('/kaggle/working')
    print('Runtime: Kaggle')
elif ON_COLAB:
    ROOT = Path('/content')
    print('Runtime: Colab')
else:
    ROOT = Path.cwd()
    print(f'Runtime: local ({ROOT})')

gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print(f'GPU: {gpu or "none detected"}')


In [ ]:
# Install core dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics==8.4.9', 'wandb==0.24.1', 'pycocotools'], check=True)
print('Core deps installed.')


In [ ]:
# Clone / update repo
import os, sys
from pathlib import Path

REPO_PATH = ROOT / 'amorphous-yolo'
if not (REPO_PATH / '.git').exists():
    os.system(f'git clone https://github.com/nipun-taneja/amorphous-yolo.git {REPO_PATH}')
    print('Cloned.')
else:
    os.system(f'git -C {REPO_PATH} pull --ff-only')
    print('Repo up to date.')
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))


In [ ]:
# WandB (optional)
import os, wandb
WANDB_PROJECT = 'amorphous-yolo-phase3'
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _k = UserSecretsClient().get_secret('WANDB_API_KEY')
        if _k: os.environ['WANDB_API_KEY'] = _k
    except Exception: pass
elif ON_COLAB:
    try:
        from google.colab import userdata
        _k = userdata.get('WANDB_API_KEY')
        if _k: os.environ['WANDB_API_KEY'] = _k
    except Exception: pass
if os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
    print(f'WandB active → {WANDB_PROJECT}')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('WandB disabled.')


## Section 2 · Constants & Dataset

In [ ]:
# Phase 3 constants & paths
import json as _json, math, time
from pathlib import Path
from datetime import datetime

PROJECT_DIR   = ROOT / 'amorphous-yolo'
# Must match notebook 03: datasets/kvasir-seg (not data/kvasir_seg)
DATASET_ROOT  = PROJECT_DIR / 'datasets' / 'kvasir-seg'
EXPERIMENTS   = ROOT / 'experiments_phase3'
ANALYSIS_DIR  = EXPERIMENTS / 'analysis'
METRICS_DIR   = EXPERIMENTS / 'metrics'
MANIFEST_PATH = EXPERIMENTS / 'manifest.json'
for _d in [EXPERIMENTS, ANALYSIS_DIR, METRICS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

EPOCHS = 20
IMGSZ  = 640
SEEDS  = [42]
DEVICE = 0

SPLIT_CONFIGS = {
    'clean': PROJECT_DIR / 'data' / 'kvasir_seg.yaml',
    'low':   PROJECT_DIR / 'data' / 'kvasir_seg_low.yaml',
    'high':  PROJECT_DIR / 'data' / 'kvasir_seg_high.yaml',
}

# Drive persistence (Colab only)
DRIVE_AVAILABLE = False
if ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_EXPERIMENTS = Path('/content/drive/MyDrive/experiments_phase3')
        DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)
        DRIVE_AVAILABLE = True
        print(f'Drive mounted: {DRIVE_EXPERIMENTS}')
    except Exception as e:
        print(f'Drive not available: {e}')
print(f'DATASET_ROOT = {DATASET_ROOT}')
print(f'EXPERIMENTS  = {EXPERIMENTS}')


In [ ]:
# Download Kvasir-SEG, convert masks→labels, build noise splits (all idempotent)
import zipfile, urllib.request, shutil, ssl, cv2, random
import numpy as np
from pathlib import Path

# ── 1. Download raw zip ────────────────────────────────────────────────────
kvasir_zip = PROJECT_DIR / 'datasets' / 'kvasir-seg.zip'
images_dir = DATASET_ROOT / 'images'
if images_dir.exists() and len(list(images_dir.glob('*.jpg'))) > 900:
    print(f'Kvasir-SEG images already present ({len(list(images_dir.glob("*.jpg")))} imgs).')
else:
    print('Downloading Kvasir-SEG (~44 MB)...')
    (PROJECT_DIR / 'datasets').mkdir(parents=True, exist_ok=True)
    ctx = ssl._create_unverified_context()
    with urllib.request.urlopen('https://datasets.simula.no/downloads/kvasir-seg.zip',
                                context=ctx) as r, open(kvasir_zip, 'wb') as f:
        shutil.copyfileobj(r, f)
    with zipfile.ZipFile(kvasir_zip, 'r') as z:
        z.extractall(PROJECT_DIR / 'datasets')
    up = PROJECT_DIR / 'datasets' / 'Kvasir-SEG'
    if up.exists(): up.rename(DATASET_ROOT)
    print(f'Extracted → {DATASET_ROOT}')

# ── 2. Convert masks → YOLO labels + train/val split ───────────────────────
TRAIN_DIR = DATASET_ROOT / 'train'
VALID_DIR = DATASET_ROOT / 'valid'

def _mask_to_yolo_bbox(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return None
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    H, W = mask.shape
    return tuple(float(np.clip(v, 0.0, 1.0)) for v in
                 [(x+w/2)/W, (y+h/2)/H, w/W, h/H])

if (TRAIN_DIR/'images').exists() and len(list((TRAIN_DIR/'images').glob('*.jpg'))) >= 800:
    print('Train/val split already exists.')
else:
    print('Converting masks → YOLO labels...')
    for d in [TRAIN_DIR, VALID_DIR]:
        (d/'images').mkdir(parents=True, exist_ok=True)
        (d/'labels').mkdir(parents=True, exist_ok=True)
    all_imgs = sorted((DATASET_ROOT/'images').glob('*.jpg'))
    random.seed(42); random.shuffle(all_imgs)
    sp = int(0.8 * len(all_imgs))
    splits = {'train': all_imgs[:sp], 'valid': all_imgs[sp:]}
    skipped = 0
    for sname, imgs in splits.items():
        for img_p in imgs:
            bb = _mask_to_yolo_bbox(DATASET_ROOT/'masks'/img_p.name)
            if bb is None: skipped += 1; continue
            cx, cy, bw, bh = bb
            shutil.copy2(img_p, DATASET_ROOT/sname/'images'/img_p.name)
            (DATASET_ROOT/sname/'labels'/f'{img_p.stem}.txt').write_text(
                f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
    print(f'Done. Skipped {skipped} blank masks.')
    print(f'Train: {len(list((TRAIN_DIR/"images").glob("*.jpg")))}  '
          f'Valid: {len(list((VALID_DIR/"images").glob("*.jpg")))}')

# ── 3. Build low/high noise val splits ────────────────────────────────────
def _jitter_label(src, dst, sigma, rng):
    lines = Path(src).read_text().strip().splitlines()
    out = []
    for line in lines:
        if not line.strip(): continue
        p = line.split()
        cx,cy,w,h = [float(v) for v in p[1:5]]
        cx = float(np.clip(cx+rng.normal(0,sigma), 0, 1))
        cy = float(np.clip(cy+rng.normal(0,sigma), 0, 1))
        w  = float(np.clip(w +rng.normal(0,sigma), 0.01, 1))
        h  = float(np.clip(h +rng.normal(0,sigma), 0.01, 1))
        out.append(f'{p[0]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    Path(dst).write_text('\n'.join(out)+'\n')

rng = np.random.default_rng(42)
for split_name, sigma in [('valid_low', 0.02), ('valid_high', 0.08)]:
    di = DATASET_ROOT/split_name/'images'
    dl = DATASET_ROOT/split_name/'labels'
    if di.exists() and len(list(di.glob('*.jpg'))) >= 200:
        print(f'{split_name}: already exists.')
        continue
    di.mkdir(parents=True, exist_ok=True)
    dl.mkdir(parents=True, exist_ok=True)
    for img_p in sorted((VALID_DIR/'images').glob('*.jpg')):
        dst_img = di/img_p.name
        if not dst_img.exists():
            try: dst_img.symlink_to(img_p)
            except: shutil.copy2(img_p, dst_img)
        src_lbl = VALID_DIR/'labels'/f'{img_p.stem}.txt'
        if src_lbl.exists():
            _jitter_label(src_lbl, dl/f'{img_p.stem}.txt', sigma, rng)
    print(f'{split_name}: {len(list(di.glob("*.jpg")))} images (sigma={sigma})')

# Verify
VALID_IMAGES_DIR = VALID_DIR / 'images'
n = len(list(VALID_IMAGES_DIR.glob('*.jpg'))) if VALID_IMAGES_DIR.exists() else 0
print(f'Val images: {n}  (expected 200)')
assert n >= 200, f'Dataset incomplete: only {n} val images found!'


In [ ]:
# Patch YAML files: rewrite 'path:' to match DATASET_ROOT on this runtime
# The repo YAMLs have /content/ (Colab) paths hardcoded — this fixes them for Kaggle/local.
import re
YAMLS_DIR = EXPERIMENTS / 'yamls'
YAMLS_DIR.mkdir(exist_ok=True)

PATCHED_SPLIT_CONFIGS = {}
for split_name, orig_yaml in SPLIT_CONFIGS.items():
    content = Path(orig_yaml).read_text()
    content = re.sub(r'^path:.*$', f'path: {DATASET_ROOT}', content, flags=re.MULTILINE)
    patched = YAMLS_DIR / Path(orig_yaml).name
    patched.write_text(content)
    PATCHED_SPLIT_CONFIGS[split_name] = patched
    print(f'Patched {split_name}: path -> {DATASET_ROOT}')

SPLIT_CONFIGS = PATCHED_SPLIT_CONFIGS
print('SPLIT_CONFIGS updated to patched YAMLs.')


## Section 3 · Loss Configs

In [ ]:
# Import loss classes
from src.losses import (
    CIoULoss, EIoULoss, ECIoULoss, AEIoULoss,
)
import ultralytics.utils.loss as _ul
_ORIG_BBOX_FWD = _ul.BboxLoss.forward

def patch_loss(loss_fn):
    import torch
    def _fwd(self, pred_dist, pred_bboxes, anchor_points, target_bboxes,
             target_scores, target_scores_sum, fg_mask):
        loss_iou = loss_fn(
            pred_bboxes[fg_mask], target_bboxes[fg_mask]
        ) * target_scores[fg_mask].sum() / target_scores_sum
        loss_dfl = torch.tensor(0.0, device=pred_dist.device)
        if self.use_dfl:
            loss_dfl = self.dfl_loss(
                pred_dist[fg_mask].view(-1, self.reg_max + 1),
                target_bboxes[fg_mask],
                anchor_points,
            ) * target_scores[fg_mask].sum() / target_scores_sum
        return loss_iou, loss_dfl
    _ul.BboxLoss.forward = _fwd

def restore_loss():
    _ul.BboxLoss.forward = _ORIG_BBOX_FWD

# Auto-select best AEIoU rigidities from Phase 2 (fallback to defaults)
import json as _json
_p2_metrics = Path('/kaggle/working/amorphous-yolo/experiments_kvasir/metrics_all_losses.json')
if not _p2_metrics.exists():
    _p2_metrics = Path('/content/amorphous-yolo/experiments_kvasir/metrics_all_losses.json')
try:
    _data = _json.loads(_p2_metrics.read_text())
    _aeiou = [(v.get('map50_95', 0), v['loss']) for v in _data.values()
              if v.get('loss', '').startswith('aeiou')]
    _top3 = sorted(_aeiou, reverse=True)[:3]
    AEIOU_RIGIDITIES = [float(n.split('r')[-1].replace('p', '.')) for _, n in _top3]
    print(f'Best AEIoU rigidities from Phase 2: {AEIOU_RIGIDITIES}')
except Exception:
    AEIOU_RIGIDITIES = [0.1, 0.3, 0.5]
    print(f'Phase 2 metrics not found. Using default rigidities: {AEIOU_RIGIDITIES}')

LOSS_CONFIGS = [
    ('ciou',  CIoULoss()),
    ('eiou',  EIoULoss()),
    ('eciou', ECIoULoss()),
] + [(f'aeiou_r{str(r).replace(".", "p")}', AEIoULoss(rigidity=r)) for r in AEIOU_RIGIDITIES]
print(f'Loss configs: {[n for n,_ in LOSS_CONFIGS]}')


In [ ]:
# Build COCO GT JSON from YOLO val labels (idempotent)
import cv2, json as _json

def build_coco_gt_json(img_dir, lbl_dir, out_path):
    if Path(out_path).exists(): return
    images, anns, ann_id = [], [], 1
    for p in sorted(Path(img_dir).glob('*.jpg')):
        img = cv2.imread(str(p))
        H, W = img.shape[:2]
        img_id = int(p.stem) if p.stem.isdigit() else hash(p.stem) % 10**6
        images.append({'id': img_id, 'file_name': p.name, 'width': W, 'height': H})
        lbl = Path(lbl_dir) / f'{p.stem}.txt'
        if not lbl.exists(): continue
        for line in lbl.read_text().strip().splitlines():
            _, cx, cy, bw, bh = map(float, line.split())
            x1, y1 = (cx-bw/2)*W, (cy-bh/2)*H
            anns.append({'id': ann_id, 'image_id': img_id, 'category_id': 1,
                          'bbox': [x1, y1, bw*W, bh*H],
                          'area': bw*W*bh*H, 'iscrowd': 0})
            ann_id += 1
    coco = {'images': images, 'annotations': anns,
            'categories': [{'id': 1, 'name': 'polyp'}]}
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Path(out_path).write_text(_json.dumps(coco))
    print(f'COCO GT JSON: {out_path} ({len(images)} imgs, {len(anns)} anns)')

COCO_GT_JSON = DATASET_ROOT / 'valid' / 'coco_gt.json'
build_coco_gt_json(
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    COCO_GT_JSON,
)


## Section 4 · Training Functions

In [ ]:
# compute_metrics_ultralytics() — persist PR + COCO AP + confusion matrix
import numpy as np, json as _json
from pathlib import Path
from ultralytics import YOLO

def compute_metrics_ultralytics(run_name, weights_path, yaml_path, force=False):
    out_dir = METRICS_DIR / run_name
    if (out_dir / 'coco_ap_suite.json').exists() and not force:
        print(f'  [SKIP metrics] {run_name}'); return
    if not Path(weights_path).exists():
        print(f'  [MISS] {weights_path}'); return
    out_dir.mkdir(exist_ok=True)
    model = YOLO(str(weights_path))
    val_res = model.val(data=str(yaml_path), verbose=False,
                        save_json=True, conf=0.001, iou=0.6)
    # PR curve
    try:
        prec = np.array(val_res.box.p).reshape(-1).tolist()
        rec  = np.array(val_res.box.r).reshape(-1).tolist()
        f1   = np.array(val_res.box.f1).reshape(-1).tolist()
    except Exception:
        prec = rec = f1 = []
    pr_data = {'precision': prec, 'recall': rec, 'f1': f1,
               'ap50': float(val_res.box.map50),
               'ap75': float(val_res.box.map75),
               'map50_95': float(val_res.box.map),
               'ap_per_iou': val_res.box.maps.tolist()}
    (out_dir / 'pr_curve.json').write_text(_json.dumps(pr_data, indent=2))
    # COCO AP suite via pycocotools
    coco_suite = {'map50_95': float(val_res.box.map),
                  'map50': float(val_res.box.map50),
                  'map75': float(val_res.box.map75),
                  'APs': None, 'APm': None, 'APl': None}
    preds = list(Path(str(val_res.save_dir)).glob('*predictions*.json'))
    if preds:
        try:
            from pycocotools.coco import COCO
            from pycocotools.cocoeval import COCOeval
            gt  = COCO(str(COCO_GT_JSON))
            dt  = gt.loadRes(str(preds[0]))
            ev  = COCOeval(gt, dt, 'bbox')
            ev.evaluate(); ev.accumulate(); ev.summarize()
            s = ev.stats
            coco_suite.update({'map50_95': float(s[0]), 'map50': float(s[1]),
                               'map75': float(s[2]), 'APs': float(s[3]),
                               'APm': float(s[4]), 'APl': float(s[5]),
                               'AR_1': float(s[6]), 'AR_10': float(s[7]),
                               'AR_100': float(s[8])})
        except Exception as e:
            print(f'  [WARN] pycocotools: {e}')
    (out_dir / 'coco_ap_suite.json').write_text(_json.dumps(coco_suite, indent=2))
    # Confusion matrix
    try:
        cm = model.validator.metrics.confusion_matrix
        mat = cm.matrix.tolist()
        conf_out = {'matrix': mat, 'class_names': ['polyp'],
                    'TP': mat[0][0], 'FN': mat[0][1] if len(mat[0])>1 else None,
                    'FP': mat[1][0] if len(mat)>1 else None}
    except Exception as e:
        conf_out = {'error': str(e)}
    (out_dir / 'confusion_matrix.json').write_text(_json.dumps(conf_out, indent=2))
    print(f'  [OK] {run_name}  mAP50={coco_suite["map50"]:.4f}  mAP50-95={coco_suite["map50_95"]:.4f}')


In [ ]:
# run_training() — shared Ultralytics trainer (YOLOv26s + RT-DETR-L)
import shutil, json as _json, time
from datetime import datetime
from ultralytics import YOLO

def _load_manifest():
    return _json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}

def _write_manifest(run_name, meta):
    m = _load_manifest(); m[run_name] = meta
    MANIFEST_PATH.write_text(_json.dumps(m, indent=2))

def _sync_to_drive(run_name):
    if not DRIVE_AVAILABLE: return
    try:
        shutil.copytree(str(EXPERIMENTS / run_name),
                        str(DRIVE_EXPERIMENTS / run_name), dirs_exist_ok=True)
    except Exception as e:
        print(f'  [DRIVE] {e}')

def run_training(arch_prefix, model_pt, loss_name, loss_fn,
                 split_name, yaml_path, seed=42, epochs=None):
    epochs = epochs or EPOCHS
    run_name = f'phase3_{arch_prefix}_{loss_name}_{split_name}_s{seed}_e{epochs}'
    run_dir  = EXPERIMENTS / run_name
    if (run_dir / 'results.csv').exists():
        print(f'[SKIP] {run_name}'); return run_dir
    local_last = run_dir / 'weights' / 'last.pt'
    resuming   = local_last.exists()
    tag = '[RESUME]' if resuming else '[START]'
    print(f'\n{"="*65}\n{tag} {run_name}\n{"="*65}')
    meta = {'arch': arch_prefix, 'loss': loss_name, 'split': split_name,
            'seed': seed, 'epochs': epochs, 'status': 'running',
            'timestamp': datetime.now().isoformat()}
    _write_manifest(run_name, meta)
    t0 = time.time()
    try:
        import os as _os
        _os.environ.update({'WANDB_PROJECT': WANDB_PROJECT, 'WANDB_NAME': run_name})
        patch_loss(loss_fn)
        if resuming:
            model = YOLO(str(local_last))
            results = model.train(resume=True)
        else:
            model = YOLO(model_pt)
            results = model.train(
                data=str(yaml_path), epochs=epochs, imgsz=IMGSZ,
                project=str(EXPERIMENTS), name=run_name,
                device=DEVICE, seed=seed, exist_ok=True,
            )
        meta['status'] = 'complete'
        meta['elapsed_sec'] = round(time.time() - t0, 1)
        try:
            import wandb as _w
            if _w.run: _w.finish()
        except Exception: pass
    except Exception as e:
        print(f'  [ERROR] {e}')
        meta['status'] = 'failed'; meta['error'] = str(e); raise
    finally:
        restore_loss(); _write_manifest(run_name, meta); _sync_to_drive(run_name)
    print(f'[DONE] {run_name}  ({meta["elapsed_sec"]:.0f}s)')
    return run_dir
print('run_training() ready.')


## Section 5 · YOLOv26s Training Loop

In [ ]:
# YOLOv26s — 18 runs (6 losses × 3 splits)
ARCH_PREFIX = 'yolo26s'
MODEL_PT    = 'yolov8s.pt'  # YOLOv26s checkpoint (same backbone class)

total, done, skipped, failed = 0, 0, 0, 0
for loss_name, loss_fn in LOSS_CONFIGS:
    for split_name, yaml_path in SPLIT_CONFIGS.items():
        total += 1
        try:
            result = run_training(ARCH_PREFIX, MODEL_PT, loss_name, loss_fn,
                                  split_name, yaml_path)
            done += 1 if result else 0
            skipped += 1 if result is None else 0
        except Exception as e:
            print(f'  [ERROR] {loss_name}/{split_name}: {e}'); failed += 1
print(f'\nYOLOv26s: {done} done, {skipped} skipped, {failed} failed / {total} total')


## Section 6 · Metrics Extraction

In [ ]:
# Compute and persist metrics for all completed YOLOv26s runs
for loss_name, _ in LOSS_CONFIGS:
    for split_name, yaml_path in SPLIT_CONFIGS.items():
        run_name = f'phase3_{ARCH_PREFIX}_{loss_name}_{split_name}_s42_e{EPOCHS}'
        weights  = EXPERIMENTS / run_name / 'weights' / 'best.pt'
        compute_metrics_ultralytics(run_name, weights, yaml_path)
print('Metrics done.')


## Section 7 · Save & Output

In [ ]:

# Save shard summary CSV
import pandas as pd, json as _json

rows = []
for run_dir in sorted(EXPERIMENTS.glob(f'phase3_{ARCH_PREFIX}_*')):
    run_name = run_dir.name
    csv_path = run_dir / 'results.csv'
    if not csv_path.exists(): continue
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    parts = run_name.split('_')
    # run format: phase3_{arch}_{loss...}_{split}_s{seed}_e{epochs}
    # loss may contain underscores (e.g. aeiou_r0p1) so join parts[2:-3]
    loss_name  = '_'.join(parts[2:-3])
    split_name = parts[-3]
    rows.append({
        'run': run_name,
        'arch': ARCH_PREFIX,
        'loss': loss_name,
        'split': split_name,
        'map50':    float(last.get('metrics/mAP50(B)', 0) or 0),
        'map50_95': float(last.get('metrics/mAP50-95(B)', 0) or 0),
    })

summary_csv = EXPERIMENTS / f'summary_{ARCH_PREFIX}.csv'
pd.DataFrame(rows).to_csv(summary_csv, index=False)
print(f'Summary saved: {summary_csv}  ({len(rows)} runs)')
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:

# ── OUTPUT INSTRUCTIONS ──────────────────────────────────────────────────────
# After this notebook finishes:
#
# On Kaggle:
#   1. Go to this notebook's Output tab
#   2. Click "+ New Dataset" → name it  phase3-{arch}-results
#   3. In 06d_analysis.ipynb, add this dataset as an Input
#      Path will be: /kaggle/input/phase3-{arch}-results/
#
# On Colab:
#   All outputs already synced to Drive → experiments_phase3/
#   Run 06d directly without any extra steps.
print('Shard complete. See comment above for output publishing instructions.')
